In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display

In [2]:
from pathlib import Path
import pandas as pd
import re


def resolve_root() -> Path:
    cwd = Path.cwd().resolve()

    # Notebook opened from inside Restricted_model_extended.
    if (cwd / "mps_files_infinite").is_dir() or (cwd / "mps_files_finite").is_dir():
        return cwd

    # Notebook opened from repository root.
    if (cwd / "Restricted_model_extended").is_dir():
        return cwd / "Restricted_model_extended"

    # Fallback: walk up until Restricted_model_extended is found.
    for parent in cwd.parents:
        candidate = parent / "Restricted_model_extended"
        if candidate.is_dir():
            return candidate

    return cwd


ROOT = resolve_root()
print("CWD:", Path.cwd())
print("ROOT:", ROOT)

MODEL_FOLDERS = sorted(
    p.name
    for p in ROOT.iterdir()
    if p.is_dir() and p.name.startswith("mps_files") and (p / "correlators_plots").is_dir()
)
print("Model folders with correlator plots:", MODEL_FOLDERS)

# Filename pattern for the new naming scheme:
#   {geom}_alpha{alpha}_r{r}_U{U}_mu{mu}_t{t}_cons{cons}_chi{chi}_L{L}_Nuc{Nuc}_nmax{nmax}_{tail}.png
FNAME_PLOT_RE = re.compile(
    r"(?P<geom>iMPS|fMPS)"
    r"_alpha(?P<alpha>\d+)"
    r"_r(?P<r>\d+)"
    r"_U(?P<U>[^_]+)"
    r"_mu(?P<mu>[^_]+)"
    r"_t(?P<t>[^_]+)"
    r"_cons(?P<cons>[^_]+)"
    r"_chi(?P<chi>\d+)"
    r"_L(?P<L>\d+)"
    r"_Nuc(?P<Nuc>\d+)"
    r"_nmax(?P<nmax>\d+)"
    r"_(?P<tail>.+?)\.png$"
)


def parse_numeric(tag: str) -> float:
    return float(tag.replace("m", "-").replace("p", "."))


def extract_corr_family(tail: str) -> str:
    families = ["density_density", "dipole_density_profile", "single_particle", "density", "current", "dipole"]
    for fam in families:
        if tail.startswith(fam):
            return fam
    return tail.split("_")[0]


records = []

for model_dirname in MODEL_FOLDERS:
    png_dir = ROOT / model_dirname / "correlators_plots"
    for f in png_dir.glob("*.png"):
        m = FNAME_PLOT_RE.match(f.name)
        if not m:
            print("Skipping unparsable:", f.name)
            continue

        gd = m.groupdict()
        tail = gd["tail"]
        family = extract_corr_family(tail)

        records.append(
            dict(
                path_png=f,
                model_folder=model_dirname,
                geom=gd["geom"],
                alpha=int(gd["alpha"]),
                r=int(gd["r"]),
                U=float(gd["U"]),
                mu=parse_numeric(gd["mu"]),
                t=parse_numeric(gd["t"]),
                conserve=gd["cons"],
                chi=int(gd["chi"]),
                L=int(gd["L"]),
                N_uc=int(gd["Nuc"]),
                n_max=int(gd["nmax"]),
                corr_type=family,
                plot_type=tail,
            )
        )

df = pd.DataFrame(records)
print("Total PNG records:", len(df))
if not df.empty:
    print(df.head())
    print(df.columns.tolist())
else:
    print("No PNG files found. Check ROOT and correlators_plots folders.")


CWD: c:\Users\Polito\Documents\Quantum Engineering\Quantum Engineering\Research\Quantum-Many-Body-Systems\Codes\Restricted_model_extended
ROOT: C:\Users\Polito\Documents\Quantum Engineering\Quantum Engineering\Research\Quantum-Many-Body-Systems\Codes\Restricted_model_extended
Model folders with correlator plots: ['mps_files_finite', 'mps_files_infinite']
Total PNG records: 3732
                                            path_png      model_folder  geom  \
0  C:\Users\Polito\Documents\Quantum Engineering\...  mps_files_finite  fMPS   
1  C:\Users\Polito\Documents\Quantum Engineering\...  mps_files_finite  fMPS   
2  C:\Users\Polito\Documents\Quantum Engineering\...  mps_files_finite  fMPS   
3  C:\Users\Polito\Documents\Quantum Engineering\...  mps_files_finite  fMPS   
4  C:\Users\Polito\Documents\Quantum Engineering\...  mps_files_finite  fMPS   

   alpha  r    U    mu     t conserve  chi   L  N_uc  n_max        corr_type  \
0      0  1  1.0  0.95  0.11   dipole  500  80   160      

In [3]:
import ipywidgets as widgets
from IPython.display import display, Image

if df.empty:
    raise RuntimeError("No correlator images found. Check the ROOT path printed above.")

# --- Dropdown options ---
model_options = ["ALL"] + sorted(df["model_folder"].unique())
corr_options  = ["ALL"] + sorted(df["corr_type"].unique())
cons_options  = ["ALL"] + sorted(df["conserve"].unique())
nmax_options  = ["ALL"] + sorted(df["n_max"].unique())
alpha_options = ["ALL"] + sorted(df["alpha"].unique())
r_options     = ["ALL"] + sorted(df["r"].unique())

dd_model = widgets.Dropdown(options=model_options, description="Model:",     value="ALL")
dd_corr  = widgets.Dropdown(options=corr_options,  description="Correlator:", value="ALL")
dd_cons  = widgets.Dropdown(options=cons_options,  description="Conserve:",  value="ALL")
dd_nmax  = widgets.Dropdown(options=nmax_options,  description="n_max:",     value="ALL")
dd_alpha = widgets.Dropdown(options=alpha_options, description="alpha:",     value="ALL")
dd_r     = widgets.Dropdown(options=r_options,     description="r:",         value="ALL")

dd_N_uc = widgets.Dropdown(options=["ALL"] + sorted(df["N_uc"].unique()), description="N particles:", value="ALL")
dd_chi  = widgets.Dropdown(options=["ALL"] + sorted(df["chi"].unique()),  description="chi_max:",     value="ALL")

dd_U  = widgets.Dropdown(options=["ALL"] + sorted(df["U"].unique()),      description="U:",           value="ALL")
dd_mu = widgets.Dropdown(description="mu:")
dd_t  = widgets.Dropdown(description="t:")

dd_kind = widgets.Dropdown(
    options=[("All", "all"), ("Correlators only", "corr"), ("FFT only", "fft")],
    description="Plots:",
    value="all",
)

out = widgets.Output()


In [4]:
def _base_mask(include_nmax=False, include_alpha=False, include_r=False,
               include_N_uc=False, include_chi=False,
               include_U=False, include_mu=False, include_t=False):
    mask = pd.Series(True, index=df.index)

    if dd_corr.value  != "ALL": mask &= df["corr_type"].eq(dd_corr.value)
    if dd_model.value != "ALL": mask &= df["model_folder"].eq(dd_model.value)
    if dd_cons.value  != "ALL": mask &= df["conserve"].eq(dd_cons.value)

    if include_nmax  and dd_nmax.value  != "ALL": mask &= df["n_max"].eq(dd_nmax.value)
    if include_alpha and dd_alpha.value != "ALL": mask &= df["alpha"].eq(dd_alpha.value)
    if include_r     and dd_r.value     != "ALL": mask &= df["r"].eq(dd_r.value)
    if include_N_uc  and dd_N_uc.value  != "ALL": mask &= df["N_uc"].eq(dd_N_uc.value)
    if include_chi   and dd_chi.value   != "ALL": mask &= df["chi"].eq(dd_chi.value)
    if include_U     and dd_U.value     != "ALL": mask &= df["U"].eq(dd_U.value)
    if include_mu    and dd_mu.value is not None: mask &= df["mu"].eq(dd_mu.value)
    if include_t     and dd_t.value not in (None, "ALL"): mask &= df["t"].eq(dd_t.value)

    return mask


def update_nmax_options(*_):
    nmaxs = sorted(df.loc[_base_mask(), "n_max"].unique())
    cur = dd_nmax.value
    dd_nmax.options = ["ALL"] + nmaxs
    dd_nmax.value = cur if cur in ["ALL"] + nmaxs else "ALL"


def update_alpha_options(*_):
    alphas = sorted(df.loc[_base_mask(include_nmax=True), "alpha"].unique())
    cur = dd_alpha.value
    dd_alpha.options = ["ALL"] + alphas
    dd_alpha.value = cur if cur in ["ALL"] + alphas else "ALL"


def update_r_options(*_):
    rs = sorted(df.loc[_base_mask(include_nmax=True, include_alpha=True), "r"].unique())
    cur = dd_r.value
    dd_r.options = ["ALL"] + rs
    dd_r.value = cur if cur in ["ALL"] + rs else "ALL"


def update_N_uc_options(*_):
    nucs = sorted(df.loc[_base_mask(include_nmax=True, include_alpha=True, include_r=True), "N_uc"].unique())
    cur = dd_N_uc.value
    dd_N_uc.options = ["ALL"] + nucs
    dd_N_uc.value = cur if cur in ["ALL"] + nucs else "ALL"


def update_chi_options(*_):
    chis = sorted(df.loc[_base_mask(include_nmax=True, include_alpha=True, include_r=True, include_N_uc=True), "chi"].unique())
    cur = dd_chi.value
    dd_chi.options = ["ALL"] + chis
    dd_chi.value = cur if cur in ["ALL"] + chis else "ALL"


def update_U_options(*_):
    Us = sorted(df.loc[
        _base_mask(include_nmax=True, include_alpha=True, include_r=True,
                   include_N_uc=True, include_chi=True),
        "U"
    ].unique())
    cur = dd_U.value
    dd_U.options = ["ALL"] + Us
    dd_U.value = cur if cur in ["ALL"] + Us else "ALL"


def update_mu_options(*_):
    mus = sorted(df.loc[
        _base_mask(include_nmax=True, include_alpha=True, include_r=True,
                   include_N_uc=True, include_chi=True, include_U=True),
        "mu"
    ].unique())
    dd_mu.options = mus
    if mus and dd_mu.value not in mus:
        dd_mu.value = mus[0]


def update_t_options(*_):
    if dd_mu.value is None:
        dd_t.options = []
        return
    ts = sorted(df.loc[
        _base_mask(include_nmax=True, include_alpha=True, include_r=True,
                   include_N_uc=True, include_chi=True, include_U=True, include_mu=True),
        "t"
    ].unique())
    cur = dd_t.value
    dd_t.options = ["ALL"] + ts
    dd_t.value = cur if cur in ["ALL"] + ts else "ALL"


In [5]:
def show_plots(*_):
    with out:
        out.clear_output()
        if dd_mu.value is None:
            print("No selection.")
            return

        subset = df[_base_mask(include_nmax=True, include_alpha=True, include_r=True,
                               include_N_uc=True, include_chi=True, include_U=True,
                               include_mu=True, include_t=True)]
        subset = subset.sort_values(
            ["model_folder", "alpha", "r", "corr_type", "conserve", "plot_type", "chi", "U", "t"]
        )

        if subset.empty:
            print("No matching plots.")
            return

        if dd_kind.value == "fft":
            subset = subset[subset["plot_type"].str.contains("FFT", na=False)]
        elif dd_kind.value == "corr":
            subset = subset[~subset["plot_type"].str.contains("FFT", na=False)]

        if subset.empty:
            print("No matching plots for this choice of 'Plots:'.")
            return

        for _, row in subset.iterrows():
            tag = (
                f"{row['model_folder']} | alpha={row['alpha']} r={row['r']} "
                f"| corr={row['corr_type']} | cons={row['conserve']} "
                f"| {row['plot_type']} "
                f"| U={row['U']} mu={row['mu']} t={row['t']} "
                f"chi={row['chi']} L={row['L']} N_uc={row['N_uc']} nmax={row['n_max']}"
            )
            print(tag)
            display(Image(filename=str(row["path_png"])))


In [ ]:
# Cascade: model/corr/cons -> nmax -> alpha -> r -> N_uc -> chi -> U -> mu -> t
top_dropdowns = [dd_model, dd_corr, dd_cons]

for dd in top_dropdowns:
    dd.observe(update_nmax_options, "value")
    dd.observe(update_alpha_options, "value")
    dd.observe(update_r_options, "value")
    dd.observe(update_N_uc_options, "value")
    dd.observe(update_chi_options, "value")
    dd.observe(update_U_options, "value")
    dd.observe(update_mu_options, "value")
    dd.observe(update_t_options, "value")
    dd.observe(show_plots, "value")

dd_nmax.observe(update_alpha_options, "value")
dd_nmax.observe(update_r_options, "value")
dd_nmax.observe(update_N_uc_options, "value")
dd_nmax.observe(update_chi_options, "value")
dd_nmax.observe(update_U_options, "value")
dd_nmax.observe(update_mu_options, "value")
dd_nmax.observe(update_t_options, "value")
dd_nmax.observe(show_plots, "value")

dd_alpha.observe(update_r_options, "value")
dd_alpha.observe(update_N_uc_options, "value")
dd_alpha.observe(update_chi_options, "value")
dd_alpha.observe(update_U_options, "value")
dd_alpha.observe(update_mu_options, "value")
dd_alpha.observe(update_t_options, "value")
dd_alpha.observe(show_plots, "value")

dd_r.observe(update_N_uc_options, "value")
dd_r.observe(update_chi_options, "value")
dd_r.observe(update_U_options, "value")
dd_r.observe(update_mu_options, "value")
dd_r.observe(update_t_options, "value")
dd_r.observe(show_plots, "value")

dd_N_uc.observe(update_chi_options, "value")
dd_N_uc.observe(update_U_options, "value")
dd_N_uc.observe(update_mu_options, "value")
dd_N_uc.observe(update_t_options, "value")
dd_N_uc.observe(show_plots, "value")

dd_chi.observe(update_U_options, "value")
dd_chi.observe(update_mu_options, "value")
dd_chi.observe(update_t_options, "value")
dd_chi.observe(show_plots, "value")

dd_U.observe(update_mu_options, "value")
dd_U.observe(update_t_options, "value")
dd_U.observe(show_plots, "value")

dd_mu.observe(update_t_options, "value")
dd_mu.observe(show_plots, "value")
dd_t.observe(show_plots, "value")
dd_kind.observe(show_plots, "value")

update_nmax_options()
update_alpha_options()
update_r_options()
update_N_uc_options()
update_chi_options()
update_U_options()
update_mu_options()
update_t_options()
show_plots()

ui = widgets.VBox(
    [
        widgets.HBox([dd_model, dd_corr, dd_cons, dd_nmax, dd_kind]),
        widgets.HBox([dd_alpha, dd_r, dd_N_uc, dd_chi]),
        widgets.HBox([dd_U, dd_mu, dd_t]),
        out,
    ]
)

display(ui)


: 